In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from google.genai import Client
client = Client()

In [3]:
from pydantic import BaseModel
class HumanMessage(BaseModel):
    role: str = "user"
    content: str

class AIMessage(BaseModel):
    role: str = "assistant"
    content: str

In [ ]:
from agent.state import (
    OverallState, 
    ReflectionState,
    QueryGenerationState, 
    WebSearchState, 
    SearchStateOutput) 
from agent.configuration import Configuration
from agent.tools_and_schemas import (
    SearchQueryList, 
    Reflection)
from agent.prompts import (
    get_current_date,
    query_writer_instructions,
    web_searcher_instructions,
    reflection_instructions,
    answer_instructions,
)
from agent.utils import (
    get_citations,
    get_research_topic,
    insert_citation_markers,
    resolve_urls,
)

In [ ]:
class WebSearchAgent:
    def __init__(self, client: Client):
        self.client = client
        self.config = Configuration()
        self.state = OverallState(
            messages=[],
            search_query=[],
            web_research_result=[],
            sources_gathered=[],
            initial_search_query_count=0,
            max_research_loops=0,
            research_loop_count=0,
            reasoning_model=None,
        )

    def generate(self, model: str, query: str) -> str:
        response = self.client.models.generate_content(
            model=model, 
            contents=query
        )
        
        ai_message = AIMessage(content=response.text)
        self.state["messages"].append(ai_message)
        return response.text

    def generate_structured(self, model: str, query: str, schema: str) -> str:
        response = self.client.models.generate_content(
            model=model, 
            contents=query,
            config={
                "response_mime_type": "application/json",
                "response_schema": schema,
            },
        )
        
        ai_message = AIMessage(content=response.text)
        self.state["messages"].append(ai_message)
        return response.parsed

    def run(self, state: OverallState) -> str:
        # check for custom initial search query count
        if state.get("initial_search_query_count") is None:
            self.state["initial_search_query_count"] = self.config.number_of_initial_queries
        else:
            self.state["initial_search_query_count"] = state["initial_search_query_count"]

        self.state["messages"].append(state["messages"])
        
        queries = self.generate_query()
        self.continue_to_web_research(queries)
        reflection = self.reflection()
        eval = self.evaluate_research(reflection)

        while True:
            if eval == "finalize_answer":
                answer = self.finalize_answer()
                return answer
            elif eval == "need more web research":
                reflection = self.reflection()
                eval = self.evaluate_research(reflection)
                

    def generate_query(self) -> QueryGenerationState:
               
        model = self.config.query_generator_model

        # Format the prompt
        current_date = get_current_date()
        formatted_prompt = query_writer_instructions.format(
            current_date=current_date,
            research_topic=get_research_topic(self.state["messages"]),
            number_queries=self.state["initial_search_query_count"],
        )
        # Generate the search queries
        result = self.generate_structured(model, formatted_prompt, SearchQueryList)
        self.state["search_query"] = result.query
        return {"search_query": result.query}

    def continue_to_web_research(self, state: QueryGenerationState):
        
        for idx, search_query in enumerate(state["search_query"]):
            s = WebSearchState(
                search_query=search_query,
                id=idx
            )
            print("web_research", s)
                        
            self.web_research(s)
                             
                
    def web_research(self, state: WebSearchState):
       
        formatted_prompt = web_searcher_instructions.format(
            current_date=get_current_date(),
            research_topic=state["search_query"],
        )

        # Uses the google genai client as the langchain client doesn't return grounding metadata
        response = self.client.models.generate_content(
            model=self.config.query_generator_model,
            contents=formatted_prompt,
            config={
                "tools": [{"google_search": {}}],
                "temperature": 0,
            },
        )
        # resolve the urls to short urls for saving tokens and time
        resolved_urls = resolve_urls(
            response.candidates[0].grounding_metadata.grounding_chunks, state["id"]
        )
        # Gets the citations and adds them to the generated text
        citations = get_citations(response, resolved_urls)
        modified_text = insert_citation_markers(response.text, citations)
        sources_gathered = [item for citation in citations for item in citation["segments"]]
        
        self.state["sources_gathered"] = self.state["sources_gathered"] + sources_gathered
        self.state["search_query"] = self.state["search_query"] + [state["search_query"]]
        self.state["web_research_result"] = self.state["web_research_result"] + [modified_text]
        
       
    def reflection(self) -> ReflectionState:
        
        # Increment the research loop count and get the reasoning model
        self.state["research_loop_count"] = self.state.get("research_loop_count", 0) + 1
        if self.state.get("reasoning_model") is None:
            self.state["reasoning_model"] = self.config.reflection_model
        
        reasoning_model = self.state.get("reasoning_model") 

        # Format the prompt
        current_date = get_current_date()
        formatted_prompt = reflection_instructions.format(
            current_date=current_date,
            research_topic=get_research_topic(self.state["messages"]),
            summaries="\n\n---\n\n".join(self.state["web_research_result"]),
        )
       
        result = self.generate_structured(reasoning_model, formatted_prompt, Reflection)
        
        return {
            "is_sufficient": result.is_sufficient,
            "knowledge_gap": result.knowledge_gap,
            "follow_up_queries": result.follow_up_queries,
            "research_loop_count": self.state["research_loop_count"],
            "number_of_ran_queries": len(self.state["search_query"]),
        }
    
    def evaluate_research(self, state: ReflectionState) -> str:
        
        max_research_loops = (
            state.get("max_research_loops")
            if state.get("max_research_loops") is not None
            else self.config.max_research_loops
        )
        if state["is_sufficient"] or state["research_loop_count"] >= max_research_loops:
            return "finalize_answer"
        else:
            for idx, follow_up_query in enumerate(state["follow_up_queries"]):
                s = WebSearchState(
                    search_query=follow_up_query,
                    id=state["number_of_ran_queries"] + int(idx),
                )               
                        
                self.web_research(s)
            return "need more web research"
    
    def finalize_answer(self) -> dict:
    
        reasoning_model = self.state.get("reasoning_model") or self.config.answer_model

        # Format the prompt
        current_date = get_current_date()
        formatted_prompt = answer_instructions.format(
            current_date=current_date,
            research_topic=get_research_topic(self.state["messages"]),
            summaries="\n---\n\n".join(self.state["web_research_result"]),
        )

        result = self.generate(reasoning_model, formatted_prompt)

        # Replace the short urls with the original urls and add all used urls to the sources_gathered
        unique_sources = []
        for source in self.state["sources_gathered"]:
            if source["short_url"] in result.content:
                result.content = result.content.replace(
                    source["short_url"], source["value"]
                )
                unique_sources.append(source)

        return {
            "messages": [AIMessage(content=result.content)],
            "sources_gathered": unique_sources,
        }
           
            
            
            

In [18]:
agent = WebSearchAgent(client)

In [19]:
res = agent.step(OverallState(messages=[HumanMessage(content="What is the capital of both of Korea?")]))
res

web_research {'search_query': 'capital of South Korea 2025', 'id': 0}
web_research {'search_query': 'capital of North Korea 2025', 'id': 1}


TypeError: WebSearchAgent.reflection() takes 1 positional argument but 2 were given

In [9]:
agent.state

{'messages': [HumanMessage(role='user', content='Your goal is to generate sophisticated and diverse web search queries. These queries are intended for an advanced automated web research tool capable of analyzing complex results, following links, and synthesizing information.\n\nInstructions:\n- Always prefer a single search query, only add another query if the original question requests multiple aspects or elements and one query is not enough.\n- Each query should focus on one specific aspect of the original question.\n- Don\'t produce more than 3 queries.\n- Queries should be diverse, if the topic is broad, generate more than 1 query.\n- Don\'t generate multiple similar queries, 1 is enough.\n- Query should ensure that the most current information is gathered. The current date is September 17, 2025.\n\nFormat: \n- Format your response as a JSON object with ALL two of these exact keys:\n   - "rationale": Brief explanation of why these queries are relevant\n   - "query": A list of searc

In [10]:
agent.config

Configuration(query_generator_model='gemini-2.0-flash', reflection_model='gemini-2.5-flash', answer_model='gemini-2.5-pro', number_of_initial_queries=3, max_research_loops=2)

In [21]:
a=OverallState(messages=[HumanMessage(content="What is the capital of both of Korea?")])

In [26]:
if a.get("initial_search_query_count") is None:
    print("None")

None
